# Classical ML Cheatsheet — GF Digital Manufacturing Prep

Covers: EDA & viz, preprocessing, statistical modelling, classification, regression,
clustering, anomaly detection, ensembles, hyperparameter tuning, and evaluation/interpretation —
i.e. everything in scope for the role (excludes DL, MLOps, LLMs, PySpark).

**How to use this notebook:** each section = one markdown cell (what it's for + when to use it)
followed by one code cell (commented line-by-line). Swap in your own `df`, `X`, `y` as needed.

## 0. Setup & Imports

Standard imports you'll need across almost every notebook. Not all libraries are used in every
section below — import what the section needs.

In [ ]:
import pandas as pd                      # dataframes
import numpy as np                       # numerical arrays, linear algebra
import matplotlib.pyplot as plt          # base plotting
import seaborn as sns                    # statistical plotting, built on matplotlib
import warnings
warnings.filterwarnings('ignore')        # suppress sklearn/pandas version warnings (remove when debugging)

sns.set_theme(style='whitegrid')         # consistent plot style across the notebook
pd.set_option('display.max_columns', 100)  # show more columns when previewing wide dataframes

## 1. Exploratory Data Analysis (EDA)

**Application:** always the first step. Goal is to understand shape, types, missingness, and
distributions before touching any model. On SECOM specifically: check missingness % per column
(many sensors have gaps) and class balance (pass/fail is imbalanced) before anything else.

In [ ]:
df = pd.read_csv('data.csv')             # load dataset — replace path

df.shape                                  # (n_rows, n_cols) — sanity check size
df.head()                                 # first 5 rows — visual sanity check
df.info()                                 # dtypes + non-null counts per column, catches type issues early
df.describe()                             # count/mean/std/min/quartiles/max for numeric cols

df.isnull().sum().sort_values(ascending=False)          # missing values per column, descending
(df.isnull().sum() / len(df) * 100).round(1)             # missing values as % of total rows — easier to judge severity

df.duplicated().sum()                     # count of exact duplicate rows

df['target'].value_counts()               # class counts — reveals imbalance (e.g. SECOM pass/fail)
df['target'].value_counts(normalize=True) # class counts as proportions instead of raw counts

df.nunique()                              # unique value count per column — flags near-constant columns (common in SECOM)
df.corr(numeric_only=True)                # pairwise correlation matrix for numeric features

## 2. Data Visualisation

**Application:** visual EDA to spot distributions, outliers, and relationships that summary
stats hide. Use histograms/boxplots for single variables, scatter/pairplot for relationships,
heatmaps for correlation structure.

**Key `histplot` parameters:**
- `data` — the values to plot (Series/array, or `data=df, x='col'` if also using `hue`)
- `kde` — overlays a smoothed density curve on the histogram — reveals skew/multimodality bars alone can hide
- `hue` — split the distribution by a category (e.g. compare pass/fail)
- `bins` — control bin granularity if the default `'auto'` binning looks off

### Correlation Heatmap — how to read it
Each cell = correlation coefficient (-1 to +1) between the row feature and column feature.
Diagonal is always 1 (self-correlation) — ignore it. Matrix is symmetric — only need to read half.

With `center=0`: one extreme colour = strong positive correlation, opposite extreme = strong negative
correlation, pale/white = ~0.

**What to look for:**
- Feature-feature cells near ±1 → multicollinearity risk (matters for linear models)
- Feature-target cells near ±1 → likely predictive feature
- Note: correlation only catches **linear** relationships — non-linear relationships can show near-0 here

On SECOM (591 features) this is visually unreadable — sort feature-target correlations programmatically
instead of eyeballing the full matrix.

### Pairplot — how to read it
Grid of every feature plotted against every other feature. Off-diagonal cells = scatterplots between two
features (catches relationships correlation alone misses). Diagonal cells = histogram/KDE of that single
feature (can't plot a feature against itself).

With `hue='target'`: scan for panels where colours visually separate into distinct clusters — those feature
pairs are your strongest predictors, even if neither feature alone is great individually.

Symmetric halves (bottom-left mirrors top-right) — only need to read one triangle.

Don't run on the full SECOM feature set (591 columns) — slow and unreadable. Pick a handful of
top-correlated features first.

In [ ]:
# --- Distribution of a single numeric feature ---
plt.figure(figsize=(6,4))
sns.histplot(
    df['feature1'],   # data: values to plot (Series/array, or data=df, x='col' if using hue)
    kde=True,          # kde: overlays a smoothed density curve on top of the histogram bars
    # bins=30          # bins: controls bin granularity if default 'auto' binning looks off
    # hue='target'     # hue: split the distribution by a category, e.g. compare pass/fail
)
plt.title('Distribution of feature1')
plt.show()

# --- Outlier check ---
plt.figure(figsize=(6,4))
sns.boxplot(x=df['feature1'])             # box = IQR, whiskers = 1.5*IQR, dots beyond whiskers = outliers
plt.show()

# --- Relationship between two numeric variables ---
plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x='feature1', y='feature2', hue='target')  # hue colours points by class
plt.show()

# --- Correlation heatmap (see MD above for how to read it) ---
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0)  # center=0 = neutral colour at zero correlation
plt.title('Feature correlation matrix')
plt.show()

# --- Pairplot (see MD above for how to read it) ---
sns.pairplot(df[['feature1','feature2','feature3','target']], hue='target')

# --- Class balance ---
plt.figure(figsize=(5,4))
sns.countplot(x='target', data=df)        # bar count per class — quick imbalance check
plt.show()

# --- Interactive version (Plotly) for stakeholder-facing work ---
# If import fails: conda activate dsprint && pip install plotly
import plotly.express as px
fig = px.scatter(df, x='feature1', y='feature2', color='target', hover_data=['feature3'])  # hover_data shows extra info on hover
fig.show()

## 3. Statistical Modelling & Hypothesis Testing

**Application:** used when you need to explain *why* a feature matters (p-values, confidence
intervals), not just predict. Common in fab contexts: "is this sensor's mean significantly
different between pass/fail lots?" (t-test), or "does yield differ across tools?" (ANOVA).

In [ ]:
import statsmodels.api as sm
from scipy import stats

# --- OLS regression with full statistical output (p-values, CIs, R²) ---
X_sm = sm.add_constant(df[['feature1','feature2']])   # statsmodels needs an explicit intercept column
ols_model = sm.OLS(df['target_numeric'], X_sm).fit()   # fit ordinary least squares
print(ols_model.summary())                             # coefficients, std errors, p-values, R², F-stat all in one table

# --- Two-sample t-test: is the mean of feature1 different between pass/fail? ---
group_pass = df[df['target']==0]['feature1']
group_fail = df[df['target']==1]['feature1']
t_stat, p_value = stats.ttest_ind(group_pass, group_fail, equal_var=False)  # equal_var=False = Welch's t-test, safer default (doesn't assume equal variances)
print(f't={t_stat:.3f}, p={p_value:.4f}')               # p < 0.05 conventionally = significant difference

# --- ANOVA: compare means across 3+ groups (e.g. across tool IDs) ---
f_stat, p_value = stats.f_oneway(df[df['tool']=='A']['feature1'],
                                   df[df['tool']=='B']['feature1'],
                                   df[df['tool']=='C']['feature1'])

# --- Chi-square: association between two categorical variables ---
contingency = pd.crosstab(df['category1'], df['category2'])   # cross-tab of counts
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

# --- Variance Inflation Factor: check multicollinearity before trusting regression coefficients ---
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_data = pd.DataFrame()
vif_data['feature'] = X_sm.columns
vif_data['VIF'] = [variance_inflation_factor(X_sm.values, i) for i in range(X_sm.shape[1])]  # VIF > 5-10 = problematic multicollinearity

## 4. Preprocessing Pipeline (shared by every model below)

**Application:** build this once, reuse everywhere. `Pipeline` + `ColumnTransformer` prevents
data leakage (fit only on train, transform on test) and keeps preprocessing reproducible.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=['target'])
y = df['target']

# stratify=y keeps class proportions consistent across train/test — important for imbalanced data like SECOM
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # median is robust to outliers vs mean
    ('scaler', StandardScaler())                      # zero mean, unit variance — needed for SVM/LogReg, not needed for tree models
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing categories with the mode
    ('onehot', OneHotEncoder(handle_unknown='ignore'))      # handle_unknown='ignore' prevents crashes on unseen categories at test time
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])   # applies the right pipeline to the right column subset

# --- Handling class imbalance (SECOM-relevant) ---
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # imblearn's Pipeline supports resampling steps; sklearn's does not

full_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),    # generates synthetic minority-class samples — only applied during fit, never at inference
    ('classifier', None)                   # placeholder, set per-model below
])

## 5. Classification Pipeline

**Application:** predicting a category — SECOM pass/fail is the direct example. Use
precision/recall/F1/ROC-AUC over plain accuracy when classes are imbalanced (accuracy is
misleading when 95% of samples are one class).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              precision_recall_curve, roc_curve, ConfusionMatrixDisplay)

# --- Logistic Regression (interpretable baseline) ---
logreg = Pipeline([('preprocess', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
logreg.fit(X_train, y_train)

# --- Random Forest (strong non-linear baseline, handles mixed feature scales) ---
rf = Pipeline([('preprocess', preprocessor), ('clf', RandomForestClassifier(n_estimators=200, random_state=42))])
rf.fit(X_train, y_train)

# --- XGBoost (usually best performer on tabular data like SECOM) ---
neg, pos = y_train.value_counts()
xgb = Pipeline([('preprocess', preprocessor),
                 ('clf', XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                        scale_pos_weight=neg/pos,   # upweights minority class — key for imbalance
                                        eval_metric='logloss', random_state=42))])
xgb.fit(X_train, y_train)

# --- SVM (good for smaller, cleanly-separable datasets; slow on high-dim like raw SECOM) ---
svm = Pipeline([('preprocess', preprocessor), ('clf', SVC(kernel='rbf', probability=True))])  # probability=True needed for ROC-AUC
svm.fit(X_train, y_train)

# --- Evaluation (do this for every model above) ---
y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]   # probability of the positive class, needed for ROC-AUC

print(classification_report(y_test, y_pred))            # precision/recall/F1 per class
print('ROC-AUC:', roc_auc_score(y_test, y_proba))        # threshold-independent ranking metric

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)  # visual TP/FP/FN/TN breakdown
plt.show()

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)  # useful when positive class is rare — more informative than ROC in that case
plt.plot(recalls, precisions)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.show()

## 6. Regression Pipeline

**Application:** predicting a continuous value — e.g. HDB resale price, or a continuous yield
metric. Same pipeline structure as classification, different models and metrics.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Linear Regression (interpretable baseline) ---
linreg = Pipeline([('preprocess', preprocessor), ('reg', LinearRegression())])
linreg.fit(X_train, y_train)

# --- Ridge (L2 penalty — shrinks coefficients, helps with multicollinearity) ---
ridge = Pipeline([('preprocess', preprocessor), ('reg', Ridge(alpha=1.0))])   # alpha = regularisation strength, higher = more shrinkage

# --- Lasso (L1 penalty — can shrink coefficients to exactly zero, i.e. feature selection) ---
lasso = Pipeline([('preprocess', preprocessor), ('reg', Lasso(alpha=0.1))])

# --- Random Forest / XGBoost regressors — same pattern as classifiers above ---
rf_reg = Pipeline([('preprocess', preprocessor), ('reg', RandomForestRegressor(n_estimators=200, random_state=42))])
xgb_reg = Pipeline([('preprocess', preprocessor), ('reg', XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42))])

xgb_reg.fit(X_train, y_train)
y_pred = xgb_reg.predict(X_test)

print('RMSE:', mean_squared_error(y_test, y_pred, squared=False))  # same units as target — most interpretable
print('MAE:', mean_absolute_error(y_test, y_pred))                  # average absolute error, robust to outliers vs RMSE
print('R²:', r2_score(y_test, y_pred))                              # proportion of variance explained

# --- Residual plot: check for patterns that signal a poor model fit ---
residuals = y_test - y_pred
plt.scatter(y_pred, residuals)
plt.axhline(0, color='red', linestyle='--')   # residuals should scatter randomly around 0 with no pattern
plt.xlabel('Predicted'); plt.ylabel('Residual')
plt.show()

## 7. Clustering (unsupervised)

**Application:** no target variable — find natural groupings. Use when segmenting customers,
sensors, or lots without predefined labels. K-Means needs you to choose k; DBSCAN finds it
automatically but needs distance-parameter tuning.

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# scale first — clustering is distance-based, unscaled features dominate the distance metric
X_scaled = StandardScaler().fit_transform(df[['feature1','feature2']])

# --- Elbow method: pick k by looking for the 'bend' in inertia vs k ---
inertias = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)   # n_init=10 runs k-means 10x with different seeds, keeps best
    km.fit(X_scaled)
    inertias.append(km.inertia_)   # sum of squared distances to nearest cluster centre — lower is tighter clusters
plt.plot(range(1,10), inertias, marker='o')
plt.xlabel('k'); plt.ylabel('Inertia')
plt.show()

# --- Fit K-Means with chosen k ---
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)   # returns cluster label per row

silhouette_score(X_scaled, clusters)   # -1 to 1, higher = better-separated clusters (>0.5 is generally good)

# --- DBSCAN: density-based, finds arbitrary shapes, auto-detects outliers as noise (-1 label) ---
dbscan = DBSCAN(eps=0.5, min_samples=5)   # eps = neighbourhood radius, min_samples = points needed to form a dense region
clusters_db = dbscan.fit_predict(X_scaled)

# --- Hierarchical clustering + dendrogram ---
from scipy.cluster.hierarchy import dendrogram, linkage
Z = linkage(X_scaled, method='ward')   # ward minimises within-cluster variance when merging
dendrogram(Z)
plt.show()

# --- Visualise clusters ---
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=clusters, cmap='viridis')
plt.show()

## 8. Anomaly Detection

**Application:** flagging rare/abnormal points — directly relevant to fab defect detection.
Two flavours: unsupervised (no labels, e.g. Isolation Forest) or treating it as an extreme
classification problem (if you do have fail labels, like SECOM).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

# --- Isolation Forest: isolates anomalies via random splits, they need fewer splits to isolate ---
iso = IsolationForest(contamination=0.05, random_state=42)   # contamination = expected proportion of anomalies in the data
preds = iso.fit_predict(X_scaled)   # returns 1 = normal, -1 = anomaly
anomaly_scores = iso.decision_function(X_scaled)   # lower score = more anomalous

# --- One-Class SVM: learns a boundary around 'normal' data, flags points outside it ---
ocsvm = OneClassSVM(nu=0.05, kernel='rbf')   # nu ~ expected anomaly fraction, similar role to contamination above
preds_svm = ocsvm.fit_predict(X_scaled)

# --- Local Outlier Factor: flags points with much lower density than their neighbours ---
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
preds_lof = lof.fit_predict(X_scaled)   # note: LOF has no separate .predict(), only fit_predict on training data

df['is_anomaly'] = (preds == -1)
sns.scatterplot(data=df, x='feature1', y='feature2', hue='is_anomaly')
plt.show()

## 9. Ensemble Methods (beyond single RF/XGB)

**Application:** combine multiple models to squeeze out extra performance, or to reduce
variance from any single model's quirks. Worth trying after individual models are already tuned.

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier, BaggingClassifier, AdaBoostClassifier

# --- Voting: combine predictions from several fitted models ---
voting = VotingClassifier(estimators=[
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=200, random_state=42))
], voting='soft')   # 'soft' averages predicted probabilities, 'hard' takes majority class vote — soft usually performs better
voting.fit(X_train_processed, y_train)   # note: needs already-preprocessed X (fit preprocessor separately first)

# --- Stacking: trains a meta-model on top of base models' predictions ---
stacking = StackingClassifier(estimators=[
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=200, random_state=42))
], final_estimator=LogisticRegression())   # meta-model learns how to best combine base model outputs
stacking.fit(X_train_processed, y_train)

# --- Bagging: trains many copies of a base model on bootstrap samples, averages them (this is literally what Random Forest does internally) ---
bagging = BaggingClassifier(estimator=LogisticRegression(), n_estimators=50, random_state=42)

# --- AdaBoost: sequentially reweights misclassified samples so later models focus on hard cases ---
ada = AdaBoostClassifier(n_estimators=100, random_state=42)   # conceptually similar to XGBoost but simpler/older boosting approach

## 10. Hyperparameter Tuning

**Application:** squeezing extra performance out of a chosen model. GridSearch is exhaustive
but slow; RandomizedSearch samples a subset; Optuna is smarter (Bayesian-ish) and preferred
once search spaces get large, like XGBoost's.

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import optuna

# --- GridSearchCV: exhaustively tries every combination — fine for small search spaces ---
param_grid = {'clf__n_estimators': [100, 200, 300], 'clf__max_depth': [3, 5, 7]}
grid = GridSearchCV(rf, param_grid, cv=5, scoring='f1', n_jobs=-1)   # cv=5 = 5-fold cross-validation, n_jobs=-1 uses all CPU cores
grid.fit(X_train, y_train)
grid.best_params_        # best combination found
grid.best_score_         # best cross-validated score

# --- RandomizedSearchCV: samples n_iter random combinations — faster for large spaces ---
from scipy.stats import randint, uniform
param_dist = {'clf__n_estimators': randint(100, 500), 'clf__max_depth': randint(3, 10)}
rand_search = RandomizedSearchCV(rf, param_dist, n_iter=20, cv=5, scoring='f1', random_state=42, n_jobs=-1)
rand_search.fit(X_train, y_train)

# --- Optuna: define an objective function, let it intelligently search the space ---
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),  # log=True samples more densely at the low end — appropriate for learning rates
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    }
    model = XGBClassifier(**params, random_state=42, eval_metric='logloss')
    score = cross_val_score(model, X_train_processed, y_train, cv=5, scoring='f1').mean()
    return score

study = optuna.create_study(direction='maximize')   # maximize F1; use 'minimize' for loss/error metrics
study.optimize(objective, n_trials=50)   # 50 trials is a reasonable starting budget for a small dataset like SECOM

study.best_params    # best hyperparameter combination found
study.best_value      # best score achieved

optuna.visualization.plot_optimization_history(study).show()   # see convergence over trials
optuna.visualization.plot_param_importances(study).show()      # which hyperparameters mattered most

## 11. Model Evaluation & Interpretation

**Application:** the "communicate to non-technical Fab leadership" part of the role. Feature
importance and SHAP answer "why did the model predict this" — critical for trust and adoption.

In [ ]:
from sklearn.model_selection import cross_val_score, learning_curve
import shap

# --- Cross-validation: more robust performance estimate than a single train/test split ---
scores = cross_val_score(xgb, X_train, y_train, cv=5, scoring='f1')   # returns 5 scores, one per fold
print(f'F1: {scores.mean():.3f} +/- {scores.std():.3f}')               # mean +/- std gives a sense of variability across folds

# --- Learning curve: diagnoses under/overfitting by training set size ---
train_sizes, train_scores, val_scores = learning_curve(xgb, X_train, y_train, cv=5, scoring='f1')
plt.plot(train_sizes, train_scores.mean(axis=1), label='Train')
plt.plot(train_sizes, val_scores.mean(axis=1), label='Validation')    # big gap between the two = overfitting
plt.legend(); plt.xlabel('Training set size'); plt.ylabel('F1')
plt.show()

# --- Feature importance (tree-based models only, quick but crude) ---
importances = xgb.named_steps['clf'].feature_importances_
feat_names = xgb.named_steps['preprocess'].get_feature_names_out()
pd.Series(importances, index=feat_names).sort_values(ascending=False).head(15).plot(kind='barh')
plt.show()

# --- SHAP: model-agnostic, shows direction + magnitude of each feature's contribution per prediction ---
explainer = shap.TreeExplainer(xgb.named_steps['clf'])                       # TreeExplainer is fast and exact for tree models
X_test_processed = xgb.named_steps['preprocess'].transform(X_test)
shap_values = explainer.shap_values(X_test_processed)

shap.summary_plot(shap_values, X_test_processed, feature_names=feat_names)   # global view: which features matter, and in which direction
shap.force_plot(explainer.expected_value, shap_values[0], X_test_processed[0], feature_names=feat_names)  # explains a single prediction — good for a specific fab lot's fail reason

---
### Quick reference: which metric when?

| Situation | Use |
|---|---|
| Balanced classification | Accuracy, F1 |
| Imbalanced classification (SECOM) | Precision, Recall, F1, ROC-AUC, PR-AUC |
| Regression | RMSE (same units), MAE (robust), R² (variance explained) |
| Clustering (no ground truth) | Silhouette score |
| Explaining *why*, not just predicting | statsmodels p-values, SHAP |

### Notebook maintenance
Add new cells below as questions come up during study — keep markdown-explanation +
commented-code as the pattern for anything new you add.